# 28 · Phase Collapse Under Perturbation

This notebook tests whether the zeta–GUE overlap phase survives when zeta gap structure is deliberately perturbed.

## Core idea

Previous notebooks established:
- zeta vs GUE has high overlap
- zeta vs Poisson has low overlap
- the contrast is stable across scales, features, and bootstrap resampling

Now we ask:

> Does the zeta–GUE phase collapse when the zeta gap sequence is structurally disrupted?

## Perturbation families

We compare the original zeta gaps to several perturbations:
- **baseline**: unmodified zeta gaps
- **global shuffle**: destroys all ordering
- **block shuffle**: preserves local blocks, destroys larger ordering
- **within-window shuffle**: preserves values inside local windows, destroys local order
- **multiplicative noise**: perturbs magnitudes while keeping approximate order

## Main outputs

- baseline vs perturbed phase-gap comparison
- collapse curves by perturbation strength
- phase-gap heatmaps vs window size
- bootstrap error bars for selected perturbations

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Descriptor builders

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_feature_matrix(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    if feature_family == "minimal":
        X = np.array([[window_entropy(w), unevenness(w), ratio_mean(w)] for w in windows_n], dtype=float)
    elif feature_family == "full":
        X = np.array([[window_entropy(w), unevenness(w), reversal_symmetry_score(w), ratio_mean(w), ratio_std(w)] for w in windows_n], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, X

## RBF overlap evaluator

In [ ]:
def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_pair_rbf(X0, X1):
    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    X0_train, X0_test = split_train_test(X0, frac=0.6)
    X1_train, X1_test = split_train_test(X1, frac=0.6)

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    Xtr_std, Xte_std = standardize_train_test(X_train, X_test)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    R_test = rbf_features(Xte_std, prototypes, gamma)

    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    scores = decision_function_logistic(R_test, w)
    probs = predict_proba_logistic(R_test, w)
    preds = (probs >= 0.5).astype(int)

    acc = accuracy(y_test, preds)
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]

    return {
        "accuracy": acc,
        "score_overlap": overlap_coefficient_from_hist(scores0, scores1, bins=30),
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_pair_rbf(X0, X1, n_boot=40, seed=9423):
    boot_rng = np.random.default_rng(seed)
    acc_vals = []
    score_vals = []

    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    for _ in range(n_boot):
        X0b = bootstrap_rows(X0, boot_rng)
        X1b = bootstrap_rows(X1, boot_rng)
        out = evaluate_pair_rbf(X0b, X1b)
        acc_vals.append(out["accuracy"])
        score_vals.append(out["score_overlap"])

    acc_vals = np.array(acc_vals)
    score_vals = np.array(score_vals)

    return {
        "acc_mean": float(acc_vals.mean()),
        "acc_std": float(acc_vals.std()),
        "acc_q025": float(np.quantile(acc_vals, 0.025)),
        "acc_q975": float(np.quantile(acc_vals, 0.975)),
        "score_mean": float(score_vals.mean()),
        "score_std": float(score_vals.std()),
        "score_q025": float(np.quantile(score_vals, 0.025)),
        "score_q975": float(np.quantile(score_vals, 0.975)),
    }

## Perturbation families

In [ ]:
def perturb_baseline(gaps, rng=None):
    return np.array(gaps, dtype=float).copy()

def perturb_global_shuffle(gaps, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    return np.array(gaps, dtype=float)[rng.permutation(len(gaps))]

def perturb_block_shuffle(gaps, block_size=10, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    blocks = [gaps[i:i+block_size] for i in range(0, len(gaps), block_size)]
    perm = rng.permutation(len(blocks))
    return np.concatenate([blocks[i] for i in perm])

def perturb_within_window_shuffle(gaps, block_size=10, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    out = gaps.copy()
    for i in range(0, len(gaps), block_size):
        chunk = out[i:i+block_size].copy()
        out[i:i+block_size] = chunk[rng.permutation(len(chunk))]
    return out

def perturb_multiplicative_noise(gaps, eps=0.05, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    noisy = gaps * (1.0 + eps * rng.normal(size=len(gaps)))
    return np.clip(noisy, 1e-8, None)

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 200
height_block = (0, 400)
n_boot = 40

perturbation_grid = [
    ("baseline", [0]),
    ("global_shuffle", [1]),
    ("block_shuffle", [5, 10, 20]),
    ("within_window_shuffle", [5, 10, 20]),
    ("noise", [0.01, 0.05, 0.10]),
]

## Build baseline feature sets

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]

def features_from_gaps(gaps, k=5, feature_family="minimal", n=200):
    _, X = build_feature_matrix(gaps, feature_family=feature_family, k=k)
    return X[:min(len(X), n)]

baseline_sizes = {}
for k in window_sizes:
    zX = features_from_gaps(zeta_base_gaps, k=k, feature_family=feature_family, n=sample_size)
    pX = features_from_gaps(poisson_base_gaps, k=k, feature_family=feature_family, n=sample_size)
    gX = features_from_gaps(gue_base_gaps, k=k, feature_family=feature_family, n=sample_size)
    baseline_sizes[k] = (len(zX), len(pX), len(gX))
baseline_sizes

## Main perturbation sweep

In [ ]:
perturb_results = []

for k in window_sizes:
    pX = features_from_gaps(poisson_base_gaps, k=k, feature_family=feature_family, n=sample_size)
    gX = features_from_gaps(gue_base_gaps, k=k, feature_family=feature_family, n=sample_size)

    for pert_name, strengths in perturbation_grid:
        for strength in strengths:
            local_rng = np.random.default_rng(9423 + 100*k + int(1000*strength) + len(pert_name))

            if pert_name == "baseline":
                zeta_pert = perturb_baseline(zeta_base_gaps, rng=local_rng)
            elif pert_name == "global_shuffle":
                zeta_pert = perturb_global_shuffle(zeta_base_gaps, rng=local_rng)
            elif pert_name == "block_shuffle":
                zeta_pert = perturb_block_shuffle(zeta_base_gaps, block_size=int(strength), rng=local_rng)
            elif pert_name == "within_window_shuffle":
                zeta_pert = perturb_within_window_shuffle(zeta_base_gaps, block_size=int(strength), rng=local_rng)
            elif pert_name == "noise":
                zeta_pert = perturb_multiplicative_noise(zeta_base_gaps, eps=float(strength), rng=local_rng)
            else:
                raise ValueError(pert_name)

            zX = features_from_gaps(zeta_pert, k=k, feature_family=feature_family, n=sample_size)

            out_zg = bootstrap_pair_rbf(zX, gX, n_boot=n_boot, seed=8000 + 10*k + int(1000*strength))
            out_zp = bootstrap_pair_rbf(zX, pX, n_boot=n_boot, seed=9000 + 10*k + int(1000*strength))

            perturb_results.append({
                "k": k,
                "perturbation": pert_name,
                "strength": strength,
                "zg_score_mean": out_zg["score_mean"],
                "zg_score_std": out_zg["score_std"],
                "zg_score_q025": out_zg["score_q025"],
                "zg_score_q975": out_zg["score_q975"],
                "zp_score_mean": out_zp["score_mean"],
                "zp_score_std": out_zp["score_std"],
                "zp_score_q025": out_zp["score_q025"],
                "zp_score_q975": out_zp["score_q975"],
                "zg_acc_mean": out_zg["acc_mean"],
                "zp_acc_mean": out_zp["acc_mean"],
                "phase_gap_mean": out_zg["score_mean"] - out_zp["score_mean"],
            })

len(perturb_results)

## Baseline summary

In [ ]:
baseline_rows = [r for r in perturb_results if r["perturbation"] == "baseline"]
baseline_rows

## Collapse curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)

for ax, k in zip(axes, window_sizes):
    rows = [r for r in perturb_results if r["k"] == k and r["perturbation"] != "baseline"]

    xlabels = []
    values = []

    for pert_name, strengths in perturbation_grid:
        if pert_name == "baseline":
            continue
        for s in strengths:
            matches = [r for r in rows if r["perturbation"] == pert_name and r["strength"] == s]
            if matches:
                xlabels.append(f"{pert_name}:{s}")
                values.append(matches[0]["phase_gap_mean"])

    ax.bar(range(len(values)), values)
    ax.set_xticks(range(len(values)), xlabels, rotation=45, ha="right")
    ax.set_title(f"phase gap collapse (k={k})")
    ax.set_ylabel("phase gap = overlap(zeta,GUE) - overlap(zeta,Poisson)")

plt.tight_layout()
plt.show()

## Phase-gap heatmap

In [ ]:
heatmap_rows = []
row_labels = []

for pert_name, strengths in perturbation_grid:
    if pert_name == "baseline":
        row_labels.append("baseline")
        row = []
        for k in window_sizes:
            match = next(r for r in perturb_results if r["k"] == k and r["perturbation"] == "baseline")
            row.append(match["phase_gap_mean"])
        heatmap_rows.append(row)
    else:
        for s in strengths:
            row_labels.append(f"{pert_name}:{s}")
            row = []
            for k in window_sizes:
                match = next(r for r in perturb_results if r["k"] == k and r["perturbation"] == pert_name and r["strength"] == s)
                row.append(match["phase_gap_mean"])
            heatmap_rows.append(row)

M = np.array(heatmap_rows)

plt.figure(figsize=(8.5, 7))
im = plt.imshow(M, aspect="auto", origin="upper")
plt.xticks(np.arange(len(window_sizes)), window_sizes)
plt.yticks(np.arange(len(row_labels)), row_labels)
plt.xlabel("window size k")
plt.ylabel("perturbation")
plt.title("Phase-gap heatmap")
plt.colorbar(im, label="phase gap")
plt.tight_layout()
plt.show()

## Bootstrap error bars for selected perturbations

In [ ]:
selected_rows = []
for label in [
    ("baseline", 0),
    ("global_shuffle", 1),
    ("block_shuffle", 10),
    ("within_window_shuffle", 10),
    ("noise", 0.05),
]:
    pert, s = label
    selected_rows.append((pert, s))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=False)

for ax, metric_prefix, title in zip(
    axes,
    ["zg_score", "zp_score"],
    ["zeta vs GUE overlap", "zeta vs Poisson overlap"]
):
    for pert, s in selected_rows:
        xs = []
        means = []
        lows = []
        highs = []
        for k in window_sizes:
            match = next(r for r in perturb_results if r["k"] == k and r["perturbation"] == pert and r["strength"] == s)
            xs.append(k)
            means.append(match[f"{metric_prefix}_mean"])
            lows.append(match[f"{metric_prefix}_mean"] - match[f"{metric_prefix}_q025"])
            highs.append(match[f"{metric_prefix}_q975"] - match[f"{metric_prefix}_mean"])
        ax.errorbar(xs, means, yerr=np.vstack([lows, highs]), marker="o", capsize=4, label=f"{pert}:{s}")
    ax.set_xlabel("window size k")
    ax.set_ylabel("bootstrap overlap")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Compact phase-gap table

In [ ]:
compact_summary = []
for pert_name, strengths in perturbation_grid:
    for s in strengths:
        vals = [r["phase_gap_mean"] for r in perturb_results if r["perturbation"] == pert_name and r["strength"] == s]
        compact_summary.append({
            "perturbation": pert_name,
            "strength": s,
            "mean_phase_gap_across_k": float(np.mean(vals)),
        })
compact_summary

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "n_boot": n_boot,
    "baseline_phase_gap_mean": float(np.mean([r["phase_gap_mean"] for r in perturb_results if r["perturbation"] == "baseline"])),
    "perturbation_summary": compact_summary,
}
summary

## Notes

- If the baseline phase gap is positive and substantial, while structure-destroying perturbations reduce it, then the overlap phase depends on organized zeta structure rather than generic robustness of the feature map.
- Global shuffle is the strongest destruction test; block and within-window shuffles help localize which ordering scale matters.
- Multiplicative noise probes robustness to value perturbations without fully destroying ordering.

## Next directions

A natural next notebook could do one of these:

1. repeat the collapse test for additional feature families  
2. repeat the same experiment for conditional windows  
3. add longer windows or longer-range features  
4. test train/test mismatch under perturbation